Install module tensorflow

In [1]:
!pip install tensorflow

Import libraries tensorflow, numpy and pickle

In [2]:
import pickle
import numpy
import tensorflow

Open the training and testing data files and load the data

In [4]:
with open(r"/train_qa.txt", 'rb') as fp:
	train_data = pickle.load(fp)
with open(r"/test_qa.txt", 'rb') as fp:
	test_data = pickle.load(fp)

Merge the first entry of training data and print its type

In [5]:
print(' '.join(train_data[0][0]))
type(train_data)

Mary moved to the bathroom . Sandra journeyed to the bedroom .


list

Merge the first entry of testing data and print its type

In [6]:
print(' '.join(test_data[0][0]))
type(test_data)

Mary got the milk there . John moved to the bedroom .


list

Print the length of training and testing data

In [7]:
print(len(train_data))
print(len(test_data))

10000
1000


Add training and testing data into all_data and create an empty set for vocabulary

In [83]:
all_data = train_data + test_data
vocab = set()

Update the vocabulary set by adding all unique words from each story and question in the dataset

In [115]:
for story, question, answer in all_data:
    vocab = vocab.union(set([word.lower() for word in story]))
    vocab = vocab.union(set([word.lower() for word in question]))

Add special tokens 'no' and 'yes' to the vocabulary

In [86]:
vocab.add('no')
vocab.add('yes')

Add length of vocab to vocab_len and add 1

In [87]:
vocab_len = len(vocab) + 1

# Create word_to_idx and idx_to_word mappings
word_to_idx = {word: i + 1 for i, word in enumerate(sorted(list(vocab)))}
word_to_idx['<unk>'] = 0 # Assign 0 for unknown words
idx_to_word = {i: word for word, i in word_to_idx.items()}

Calculate the maximum lengths of stories and questions in the dataset

In [88]:
max_story_len = max([len(data[0]) for data in all_data])
max_question_len = max([len(data[1]) for data in all_data])

Print the max question length

In [89]:
max_question_len

6

Import Tokenizer from tensorflow.keras.preprocessing.text

In [90]:
from tensorflow.keras.preprocessing.text import Tokenizer

Initialize tokenizer and get vocabulary size

In [91]:
tokenizer = Tokenizer()
vocab_size = len(tokenizer.word_index) + 1
print(vocab_size)

1


Import module keras

In [92]:
import keras
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

Initialize a tokenizer (top 10 words) and fit it on the sample training text

In [93]:
text  = train_data[0][0]
tokenizer = Tokenizer(num_words=10)
tokenizer.fit_on_texts(text)

Initialize lists to store stories, questions, and answers from training data

In [94]:
train_story_text = []
train_question_text = []
train_answers = []

Collect and print the first 10 stories and questions from the training dataset

In [95]:
count = 0
for story, question, answer in train_data:
    if(count == 10):
        break
    train_story_text.append(story)
    train_question_text.append(question)
    print(train_story_text)
    count += 1

[['Mary', 'moved', 'to', 'the', 'bathroom', '.', 'Sandra', 'journeyed', 'to', 'the', 'bedroom', '.']]
[['Mary', 'moved', 'to', 'the', 'bathroom', '.', 'Sandra', 'journeyed', 'to', 'the', 'bedroom', '.'], ['Mary', 'moved', 'to', 'the', 'bathroom', '.', 'Sandra', 'journeyed', 'to', 'the', 'bedroom', '.', 'Mary', 'went', 'back', 'to', 'the', 'bedroom', '.', 'Daniel', 'went', 'back', 'to', 'the', 'hallway', '.']]
[['Mary', 'moved', 'to', 'the', 'bathroom', '.', 'Sandra', 'journeyed', 'to', 'the', 'bedroom', '.'], ['Mary', 'moved', 'to', 'the', 'bathroom', '.', 'Sandra', 'journeyed', 'to', 'the', 'bedroom', '.', 'Mary', 'went', 'back', 'to', 'the', 'bedroom', '.', 'Daniel', 'went', 'back', 'to', 'the', 'hallway', '.'], ['Mary', 'moved', 'to', 'the', 'bathroom', '.', 'Sandra', 'journeyed', 'to', 'the', 'bedroom', '.', 'Mary', 'went', 'back', 'to', 'the', 'bedroom', '.', 'Daniel', 'went', 'back', 'to', 'the', 'hallway', '.', 'Sandra', 'went', 'to', 'the', 'kitchen', '.', 'Daniel', 'went', 'ba

Convert stories, questions, and answers into padded numeric vectors using a word index map

In [96]:
def vectorize_stories(
    data,
    word_idx_map,
    max_story_len=max_story_len,
    max_question_len=max_question_len
):
    X, Xq, Y = [], [], []

    for story, query, answer in data:
        x = [word_idx_map.get(word.lower(), word_idx_map['<unk>']) for word in story]
        xq = [word_idx_map.get(word.lower(), word_idx_map['<unk>']) for word in query]

        # The output layer size should be based on the number of unique answers in word_idx_map.
        # Since answers are 'yes'/'no', and these are part of word_idx_map, we use its size.
        y = np.zeros(len(word_idx_map))
        y[word_idx_map[answer.lower()]] = 1

        X.append(x)
        Xq.append(xq)
        Y.append(y)

    return (
        pad_sequences(X, maxlen=max_story_len),
        pad_sequences(Xq, maxlen=max_question_len),
        np.array(Y)
    )

Vectorize training data and check the shape of the story input tensor

In [97]:
input_train,queries_train,answers_train = vectorize_stories(train_data, word_to_idx)
input_train.shape

(10000, 156)

Display the shapes of the question and answer tensors

In [98]:
print(queries_train.shape)
print(answers_train.shape)

(10000, 6)
(10000, 38)


Vectorize test data and print the encoded story inputs

In [99]:
input_test,queries_test,answers_test = vectorize_stories(test_data, word_to_idx)
print(input_test)

[[ 0  0  0 ... 30  6  1]
 [ 0  0  0 ... 30 12  1]
 [ 0  0  0 ... 30 12  1]
 ...
 [ 0  0  0 ... 30  3  1]
 [ 0  0  0 ... 30 12  1]
 [ 0  0  0 ...  3 31  1]]


Compute the sum of answer vectors (useful to verify class distribution)

In [100]:
sum(answers_train)

array([   0.,    0.,    0.,    0.,    0.,    0.,    0.,    0.,    0.,
          0.,    0.,    0.,    0.,    0.,    0.,    0.,    0.,    0.,
          0.,    0.,    0.,    0.,    0.,    0.,    0., 4988.,    0.,
          0.,    0.,    0.,    0.,    0.,    0.,    0.,    0.,    0.,
          0., 5012.])

Import Keras model classes and layers required to build an LSTM-based neural network

In [101]:
from keras.models import Sequential, Model
from keras.layers import Embedding
from keras.layers import Input, Activation, Dense, Permute, Dropout
from keras.layers import add, dot, concatenate
from keras.layers import LSTM

Define input layers and embed stories and questions using shared vocabulary representations

In [102]:
inputseq = Input((max_story_len,))
questionseq = Input((max_question_len,))

# Input encoder M (Memory)
input_encoder_m = Embedding(vocab_len, 64)(inputseq) # vocab_len is the input dimension
input_encoder_m = Dropout(0.3)(input_encoder_m)

# Question encoder
question_encoder = Embedding(vocab_len, 64)(questionseq)
question_encoder = Dropout(0.3)(question_encoder)

Build and compile a memory-network–style LSTM model for question answering using story and question inputs

In [103]:
# This cell previously contained the model definition. It is now moved to cell CTFbBaymhwGK as per user request.

Define a simple LSTM-based encoder for story inputs using embeddings

In [104]:
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, concatenate, Activation, Permute, Dropout, add, dot
from tensorflow.keras.models import Model, Sequential

input_sequence=Input((max_story_len,))
question=Input((max_question_len,))

input_encoder_m=Sequential()
input_encoder_m.add(Embedding(input_dim=vocab_len,output_dim=64))
input_encoder_m.add(Dropout(0.3))

input_encoder_c=Sequential()
input_encoder_c.add(Embedding(input_dim=vocab_len,output_dim=max_question_len))
input_encoder_c.add(Dropout(0.3))

question_encoder=Sequential()
question_encoder.add(Embedding(input_dim=vocab_len,output_dim=64))
question_encoder.add(Dropout(0.3))

input_encoded_m=input_encoder_m(input_sequence)
input_encoded_c=input_encoder_c(input_sequence)
question_encoded=question_encoder(question)

match=dot([input_encoded_m,question_encoded],axes=(2,2))
match=Activation('softmax')(match)

response=add([match,input_encoded_c])
response=Permute((2,1))(response)

answer=Dropout(0.5)(concatenate([response,question_encoded]))
answer=LSTM(32)(answer)
answer=Dropout(0.5)(answer)
answer=Dense(vocab_len)(answer)

answer=Activation('softmax')(answer)
model=Model([input_sequence,question],answer)
model.compile(loss='categorical_crossentropy',optimizer='rmsprop',metrics=['accuracy'])

Train the model on training data and validate it using test data

In [105]:
history = model.fit([input_train, queries_train], answers_train, epochs=30, batch_size=32, validation_data=([input_test, queries_test], answers_test))

Epoch 1/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.4867 - loss: 1.2094 - val_accuracy: 0.5070 - val_loss: 0.6941
Epoch 2/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.5095 - loss: 0.7048 - val_accuracy: 0.4970 - val_loss: 0.6956
Epoch 3/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.5078 - loss: 0.6976 - val_accuracy: 0.5030 - val_loss: 0.6932
Epoch 4/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 22ms/step - accuracy: 0.4919 - loss: 0.6972 - val_accuracy: 0.4970 - val_loss: 0.6933
Epoch 5/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - accuracy: 0.4975 - loss: 0.6960 - val_accuracy: 0.5030 - val_loss: 0.6933
Epoch 6/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 24ms/step - accuracy: 0.4918 - loss: 0.6962 - val_accuracy: 0.4970 - val_loss: 0.6935
Epoch 7/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.5070 - loss: 0.6948 - val_accuracy: 0.4970 - val_loss: 0.6943
Epoch 8/30
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 21ms/step - accuracy: 0.5014 - loss: 0.6959 - val_acc

Save the trained model to a file for later use

In [106]:
file_name = 'chatbot_120_epochs.keras'
model.save(file_name)

Load saved model weights and generate predictions on the test data

In [107]:
model.load_weights(file_name)
prod_results = model.predict(([input_test, queries_test]))

32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step


Convert the first test story from a list of words into a readable sentence and print it

In [108]:
story = ' '.join(word for word in test_data[0][0])
print(story)

Mary got the milk there . John moved to the bedroom .


Convert the first test question from a list of words into a readable sentence and print it

In [109]:
query = ' '.join(word for word in test_data[0][1])
print(query)

Is John in the kitchen ?


Display the actual (ground-truth) answer for the first test sample

In [110]:
print('True Test Answer from Data is:', test_data[0][2])

True Test Answer from Data is: no


Get the index of the highest predicted probability (model’s chosen answer)

In [111]:
val_max = np.argmax(prod_results[0])

Use idx_to_word to get the word from the predicted index


In [112]:
k = idx_to_word[val_max]

Print the model’s predicted answer and its confidence score

In [113]:
print("Predicted answer is: ", k)
print("Probability of certainity was: ", prod_results[0][val_max])

Predicted answer is:  yes
Probability of certainity was:  0.52726305


Clean a question, tokenize it, and convert each word to its corresponding index

In [116]:
import re

q = 'Is John in the kitchen?'
# Remove punctuation from the question
q = re.sub(r'[?.!;,:]', '', q)
q = q.split()
q = [word_to_idx[word.lower()] for word in q]

Display the first training sample (story, question, and answer)

In [117]:
train_data[0]

(['Mary',
  'moved',
  'to',
  'the',
  'bathroom',
  '.',
  'Sandra',
  'journeyed',
  'to',
  'the',
  'bedroom',
  '.'],
 ['Is', 'Sandra', 'in', 'the', 'hallway', '?'],
 'no')